# Week 10 Mini Project: Fashion Image Classification with a CNN

**Estimated time: 35 minutes**

In this project, you will build a simple **Convolutional Neural Network (CNN)** that classifies clothing images from the **Fashion-MNIST** dataset.

Fashion-MNIST contains grayscale images of clothing items such as:

- T-shirts
- Pants
- Pullovers
- Dresses
- Coats
- Sandals
- Shirts
- Sneakers
- Bags
- Ankle boots

You will follow this workflow:

**Load images → visualize → prepare → build CNN → train → evaluate → inspect mistakes**


## Learning Goals

By the end of this project, you should be able to:

- Explain how images are stored as numerical tensors
- Build a basic CNN using TensorFlow/Keras
- Use convolution and pooling layers
- Train an image classifier
- Distinguish between **training** and **inference**
- Interpret model confidence
- Evaluate test accuracy
- Examine incorrect predictions
- Explain possible reasons for model errors
- Identify responsible-AI concerns in image classification


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)


## 1. Load Fashion-MNIST

TensorFlow provides Fashion-MNIST directly.

The dataset is already divided into:

- Training images and labels
- Test images and labels

Each image is:

- `28 × 28` pixels
- Grayscale
- One of 10 clothing classes


In [ ]:
fashion_mnist = keras.datasets.fashion_mnist

(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

class_names = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot"
]

print("Training images:", X_train.shape)
print("Training labels:", y_train.shape)
print("Test images:", X_test.shape)
print("Test labels:", y_test.shape)


## 2. Visualize Sample Images

Before building an AI model, always inspect the data.

The label is stored as a number from `0` to `9`.

The `class_names` list converts those numbers into readable names.


In [ ]:
plt.figure(figsize=(8, 8))

for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(X_train[i], cmap="gray")
    plt.title(class_names[y_train[i]])
    plt.axis("off")

plt.tight_layout()
plt.show()


### Think About the Images

1. Are the images high-resolution or low-resolution?
2. Are they grayscale or color?
3. Which classes look similar to one another?

Similar-looking classes can be harder for a model to distinguish.


## 3. Prepare the Image Data

Pixel values are currently integers from `0` to `255`.

We will scale them to the range `0` to `1`.

We also add a channel dimension.

Before:

```text
(28, 28)
```

After:

```text
(28, 28, 1)
```

The `1` means there is one grayscale channel.


In [ ]:
# Scale pixel values.
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# Add a grayscale channel dimension.
X_train = np.expand_dims(X_train, axis=-1)
X_test = np.expand_dims(X_test, axis=-1)

print("New training shape:", X_train.shape)
print("New test shape:", X_test.shape)


## 4. Build a Basic CNN

Our CNN will use:

1. **Conv2D** — learns image filters
2. **MaxPooling2D** — reduces feature-map size
3. Another **Conv2D**
4. Another **MaxPooling2D**
5. **Flatten** — converts feature maps into one long list
6. **Dense** — classification layer
7. Final **Dense** layer with 10 outputs

Why 10 outputs?

Because Fashion-MNIST has **10 clothing classes**.


In [ ]:
model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),

    # First convolution layer:
    # learns 16 different 3 × 3 filters.
    layers.Conv2D(
        16,
        kernel_size=(3, 3),
        activation="relu"
    ),

    # Reduce feature-map size.
    layers.MaxPooling2D(pool_size=(2, 2)),

    # Second convolution layer:
    # learns more complex patterns.
    layers.Conv2D(
        32,
        kernel_size=(3, 3),
        activation="relu"
    ),

    layers.MaxPooling2D(pool_size=(2, 2)),

    # Convert the final feature maps into a 1-D vector.
    layers.Flatten(),

    # Classification layer.
    layers.Dense(32, activation="relu"),

    # 10 output neurons for 10 clothing classes.
    layers.Dense(10, activation="softmax")
])

model.summary()


## 5. Compile the CNN

We use:

- **Adam** optimizer
- **Sparse categorical crossentropy** loss
- **Accuracy** as the metric

Because the labels are stored as integer class numbers, sparse categorical crossentropy is appropriate.


In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


## 6. Train the CNN

We will train for only **3 epochs** so the project fits into class time.

During training:

- Images move through the CNN
- The model makes predictions
- The loss measures errors
- The optimizer updates weights
- The process repeats

This is **training**.

Later, when the finished model predicts new images without changing its weights, that is called **inference**.


In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    verbose=1
)


## 7. Plot Training and Validation Accuracy

This graph helps us see whether the model learned during training.


In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CNN Training vs. Validation Accuracy")
plt.legend()
plt.show()


## 8. Evaluate the Model

Now test the CNN on images that were not used for training.

This is a better measure of how well the model generalizes.


In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("Test loss:", round(test_loss, 4))
print("Test accuracy:", round(test_accuracy, 4))


## 9. Make Predictions

The output layer gives 10 probabilities.

Example:

```text
[0.01, 0.00, 0.04, 0.10, ...]
```

The largest probability becomes the predicted class.

The largest probability is also often called the model's **confidence**.

Important:

> High confidence does not guarantee that a prediction is correct.


In [ ]:
probabilities = model.predict(
    X_test[:12],
    verbose=0
)

predictions = np.argmax(
    probabilities,
    axis=1
)

for i in range(12):
    print(
        f"Image {i}:",
        "Predicted =", class_names[predictions[i]],
        "| Actual =", class_names[y_test[i]],
        "| Confidence =", round(np.max(probabilities[i]), 3)
    )


## 10. Visualize Predictions

A visual inspection can reveal mistakes that are difficult to understand from accuracy alone.


In [ ]:
plt.figure(figsize=(10, 8))

for i in range(12):
    plt.subplot(3, 4, i + 1)

    plt.imshow(
        X_test[i].squeeze(),
        cmap="gray"
    )

    predicted_name = class_names[predictions[i]]
    actual_name = class_names[y_test[i]]
    confidence = np.max(probabilities[i])

    plt.title(
        f"P: {predicted_name}\n"
        f"A: {actual_name}\n"
        f"{confidence:.2f}"
    )

    plt.axis("off")

plt.tight_layout()
plt.show()


## 11. Examine Incorrect Predictions

A good AI developer does not only ask:

> "How accurate is the model?"

We should also ask:

> "What kinds of mistakes does it make?"

Run the next cell to find some incorrect predictions.


In [ ]:
# Predict a larger group of test images.
all_probabilities = model.predict(
    X_test[:1000],
    verbose=0
)

all_predictions = np.argmax(
    all_probabilities,
    axis=1
)

# Find indexes where prediction != actual label.
wrong_indexes = np.where(
    all_predictions != y_test[:1000]
)[0]

print(
    "Incorrect predictions found:",
    len(wrong_indexes)
)

print(
    "First incorrect indexes:",
    wrong_indexes[:10]
)


In [ ]:
# Display up to 6 incorrect predictions.
num_to_show = min(6, len(wrong_indexes))

plt.figure(figsize=(10, 6))

for plot_number in range(num_to_show):
    i = wrong_indexes[plot_number]

    plt.subplot(2, 3, plot_number + 1)

    plt.imshow(
        X_test[i].squeeze(),
        cmap="gray"
    )

    predicted_name = class_names[all_predictions[i]]
    actual_name = class_names[y_test[i]]
    confidence = np.max(all_probabilities[i])

    plt.title(
        f"P: {predicted_name}\n"
        f"A: {actual_name}\n"
        f"{confidence:.2f}"
    )

    plt.axis("off")

plt.tight_layout()
plt.show()


## 12. Analyze the Errors

Look at the incorrect predictions.

Answer:

1. Which classes seem easy to confuse?
2. Do some incorrect images look ambiguous even to you?
3. Did the model ever make an incorrect prediction with high confidence?
4. Why might that be dangerous in a real-world application?

Possible reasons for errors include:

- Similar-looking categories
- Low image resolution
- Limited training examples
- Unusual image shapes or positions
- The model has not learned enough useful features


## 13. Student Experiment

Choose **ONE** small experiment.

### Option A — Change the Number of Filters

Change:

```python
Conv2D(16, ...)
```

to:

```python
Conv2D(32, ...)
```

### Option B — Change the Dense Layer

Change:

```python
Dense(32, ...)
```

to:

```python
Dense(64, ...)
```

### Option C — Train One More Epoch

Change:

```python
epochs=3
```

to:

```python
epochs=4
```

Then compare the validation or test accuracy.

### Question

Did your change clearly improve the model?

Remember:

> A more complex model does not automatically mean a better model.


## 14. Responsible AI Discussion

Computer-vision systems can be useful, but they also have limitations.

Think about these issues:

### Dataset Bias
Does the training dataset represent the real-world situations where the model will be used?

### Incorrect Predictions
What happens if the model makes a serious mistake?

### Privacy
Could the images contain private or sensitive information?

### Human Oversight
Should a human review important AI decisions?

### Limitations
A model that performs well on one dataset may not perform equally well in the real world.

### Reflection

Choose one real-world application:

- Medical imaging
- Autonomous vehicles
- Wildlife monitoring
- Manufacturing inspection
- Robotics
- Satellite imagery

Explain one reason why **human oversight** would still be important.


# Final Reflection

Answer in your own words:

1. How is an image represented inside a computer?
2. What does a convolution layer do?
3. What is a feature map?
4. What does pooling do?
5. What is the difference between **training** and **inference**?
6. What does model confidence mean?
7. Why should we examine incorrect predictions instead of looking only at accuracy?
8. What is one limitation of this Fashion-MNIST model?

## Key Takeaway

A CNN learns visual patterns from images.

The complete workflow is:

**image pixels → convolution → feature maps → pooling → classification → prediction → evaluation**

A good AI developer also examines mistakes, confidence, bias, privacy, and the limitations of the model.
